This notebook processes eye-tracking data recorded with Pupil Labs Neon. It synchronizes gaze data with the scene video, estimates and visualizes scanpaths, and allows individual saccades to be inspected in more detail. Selected saccades can also be extracted as video snippets with the gaze position overlaid and optionally played back in slow motion.

In [1]:
# Import necessary libraries
import sys
import numpy as np
from pyneon import get_sample_data, Dataset, Recording
import pandas as pd


# # Download sample data (if not existing) and return the path
# sample_dir = get_sample_data("Artworks")
# print(sample_dir)

# dataset_dir = sample_dir / "Timeseries Data + Scene Video"
# dataset = Dataset(dataset_dir)
# print(dataset)

# recording = dataset[0]
# print(recording)

In [2]:
from pathlib import Path
from pyneon import Dataset

# # ── Replace this with the path to your own Neon export directory ──
# dataset_dir = Path("/Users/biancaschmid/Desktop/Spatial-Navigation-Group-1")

# # Load the dataset
# dataset = Dataset(dataset_dir)
# print(dataset)

# # Grab the first recording
# recording = dataset[0]
# print(recording)

In [3]:
# Point this at the folder containing your gaze.csv, world.mp4, etc.
recording_dir = Path("/Users/biancaschmid/Desktop/Spatial-Navigation-Group-1/Real_world_data")

# Load the single recording
recording = Recording(recording_dir)
print(recording)


Recording ID: 8ef60243-2690-4dac-9837-d7b0a844badb
Wearer ID: 85c31f9e-6ef0-4568-837b-ac07c0127633
Wearer name: osnabrück_plaza
Recording start time: 2025-03-19 14:56:20.204000
Recording duration: 2482.597s
                  exist                   filename                                                                                              path
3d_eye_states     False                       None                                                                                              None
blinks             True                 blinks.csv                 /Users/biancaschmid/Desktop/Spatial-Navigation-Group-1/Real_world_data/blinks.csv
events             True                 events.csv                 /Users/biancaschmid/Desktop/Spatial-Navigation-Group-1/Real_world_data/events.csv
fixations          True              fixations.csv              /Users/biancaschmid/Desktop/Spatial-Navigation-Group-1/Real_world_data/fixations.csv
gaze               True                   gaze.

In [4]:
# Map gaze data to the video timestamps
synced_gaze = recording.sync_gaze_to_video(overwrite=True)

# Inspect the mapped gaze data
print(synced_gaze.data.tail())

                     gaze x [px]  gaze y [px]  worn  fixation id  blink id  \
timestamp [ns]                                                               
1742395062646322222   839.738429   331.027571     1         6572      <NA>   
1742395062679666666   840.689500   331.602000     1         6572      <NA>   
1742395062713000000   839.054571   333.697857     1         6572      <NA>   
1742395062746344444   838.820000   331.784571     1         6572      <NA>   
1742395062779688888   840.908167   330.934000     1         6572      <NA>   

                     azimuth [deg]  elevation [deg]  frame_idx  
timestamp [ns]                                                  
1742395062646322222       1.219618        17.686574      74434  
1742395062679666666       1.282395        17.649758      74435  
1742395062713000000       1.173609        17.516298      74436  
1742395062746344444       1.158614        17.638418      74437  
1742395062779688888       1.297058        17.692366      74438 

In [5]:
# Estimate the scanpath based on the mapped gaze data
scanpath = recording.estimate_scanpath()

# Inspect the estimated scanpath
print(scanpath.data.head())

Estimating scanpath: 100%|██████████| 74439/74439 [13:20<00:00, 92.95it/s] 


                                                             fixations  \
timestamp [ns]                                                           
1742392580204000000    fixation id gaze x [px] gaze y [px] fixation...   
1742392580254000000    fixation id gaze x [px] gaze y [px] fixation...   
1742392580304000000    fixation id gaze x [px] gaze y [px] fixation...   
1742392580354000000    fixation id gaze x [px] gaze y [px] fixation...   
1742392580404000000    fixation id gaze x [px] gaze y [px] fixation...   

                     frame_idx  
timestamp [ns]                  
1742392580204000000          0  
1742392580254000000          1  
1742392580304000000          2  
1742392580354000000          3  
1742392580404000000          4  


In [6]:
# produce a full-width string of the first 5 rows, no truncation
print(scanpath.data.head().to_string())

                                                                                                                       fixations  frame_idx
timestamp [ns]                                                                                                                             
1742392580204000000    fixation id gaze x [px] gaze y [px] fixation status
0        <NA>         NaN         NaN            <NA>          0
1742392580254000000    fixation id gaze x [px] gaze y [px] fixation status
1        <NA>         NaN         NaN            <NA>          1
1742392580304000000    fixation id gaze x [px] gaze y [px] fixation status
2        <NA>         NaN         NaN            <NA>          2
1742392580354000000    fixation id gaze x [px] gaze y [px] fixation status
3        <NA>         NaN         NaN            <NA>          3
1742392580404000000    fixation id gaze x [px] gaze y [px] fixation status
4        <NA>         NaN         NaN            <NA>          4


In [8]:
# print fixations when column frame_idx is 1334. Frame_idx is not the idx of the dataframe, but the index of the video frame.
print(scanpath.data.loc[scanpath.data["frame_idx"] == 2000, "fixations"].values[0])

   fixation id  gaze x [px]  gaze y [px] fixation status
0          172   725.646286   408.615571          during
1          171   862.553528    376.55069         tracked
2          170   823.949524   311.566193         tracked
3          169   826.154907   298.907654         tracked
4          168   804.116089   359.618408         tracked
5          167   874.948975   244.811127         tracked
6          166   873.266602   274.559998         tracked
7          165   815.155334   379.021454         tracked
8          164   662.792175   510.012665         tracked
9          163   757.414734   353.462128         tracked
10         162   852.153809    293.64856         tracked
11         161   839.476868   297.859039         tracked
12         160   848.080261   309.882599         tracked
13         159   693.489807   459.176361         tracked
14         158   571.312561   493.661224         tracked
15         157   578.062805   603.290466         tracked
16         156   799.434082   7

In [9]:

# Overlay the scanpath on the video and show the output
recording.overlay_scanpath(show_video=True)

Loading saved scanpath from /Users/biancaschmid/Desktop/Spatial-Navigation-Group-1/Real_world_data/derivatives/scanpath.pkl
Resetting video...


Plotting scanpath on scene video: 100%|██████████| 74439/74439 [37:03<00:00, 33.47it/s]  


In [2]:
import os
import pandas as pd

# ——— CONFIG ———
DATA_DIR    = '/Users/biancaschmid/Desktop/Spatial-Navigation-Group-1/Real_world_data'
SACCADE_ID  = 766     # which saccade to inspect
MARGIN_MS   = 50      # milliseconds before/after to include
# ——————————

# 1) Load gaze and saccade tables
gaze_df = pd.read_csv(os.path.join(DATA_DIR, 'gaze.csv'))
sac_df  = pd.read_csv(os.path.join(DATA_DIR, 'saccades.csv'))

# 2) Locate saccade 766 and its timestamps (in ns)
sac_event = sac_df[sac_df['saccade id'] == SACCADE_ID].iloc[0]
start_ts  = sac_event['start timestamp [ns]']
end_ts    = sac_event['end timestamp [ns]']

# 3) Compute margin in nanoseconds
margin_ns = MARGIN_MS * 1_000_000

# 4) Filter gaze samples in [start–margin, end+margin]
mask = (
    (gaze_df['timestamp [ns]'] >= start_ts - margin_ns) &
    (gaze_df['timestamp [ns]'] <= end_ts   + margin_ns)
)
traj = gaze_df.loc[mask, ['timestamp [ns]', 'gaze x [px]', 'gaze y [px]']]

# 5) Optionally convert timestamp to seconds relative to saccade start
traj['time_rel_s'] = (traj['timestamp [ns]'] - start_ts) / 1e9

# 6) Print the trajectory
print(traj.to_string(index=False))



     timestamp [ns]  gaze x [px]  gaze y [px]  time_rel_s
1742392905571145145      945.320      391.618   -0.045121
1742392905576141145      939.962      393.640   -0.040125
1742392905581143145      937.034      388.877   -0.035123
1742392905586140145      931.799      388.622   -0.030126
1742392905591266145      926.690      387.705   -0.025000
1742392905596266145      921.632      385.825   -0.020000
1742392905601265145      917.464      381.851   -0.015001
1742392905606265145      915.096      386.945   -0.010001
1742392905611266145      909.742      384.148   -0.005000
1742392905616266145      905.295      381.932    0.000000
1742392905621265145      907.073      383.858    0.004999
1742392905626266145      919.517      391.569    0.010000
1742392905631266145      941.460      398.251    0.015000
1742392905636266145      972.036      388.170    0.020000
1742392905641265145     1009.866      394.519    0.024999
1742392905646266145     1037.186      401.059    0.030000
17423929056512

In [3]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
from pyneon import Recording

# ——— CONFIG ———
DATA_DIR      = '/Users/biancaschmid/Desktop/Spatial-Navigation-Group-1/Real_world_data'
OUTPUT_VIDEO  = '/Users/biancaschmid/Desktop/saccade_766_snippet.mp4'
SACCADE_ID    = 766     # which saccade to highlight
MARGIN_MS     = 100     # milliseconds before/after the saccade
# ——————————

# 1) Load saccades and world‐frame timestamps
sac_df = pd.read_csv(os.path.join(DATA_DIR, 'saccades.csv'))
wt_df  = pd.read_csv(os.path.join(DATA_DIR, 'world_timestamps.csv'))

# 2) Extract the timestamps (ns) for saccade 766
evt    = sac_df[sac_df['saccade id'] == SACCADE_ID].iloc[0]
start_ts = evt['start timestamp [ns]']
end_ts   = evt['end timestamp [ns]']

# 3) Find nearest video‐frame indices
#    world_timestamps.csv has one row per frame, in order
#    so its DataFrame index is the frame number
wt = wt_df['timestamp [ns]']
start_frame = int((wt - start_ts).abs().idxmin())
end_frame   = int((wt - end_ts  ).abs().idxmin())

# 4) Compute margin in frames
#    we'll open the video first to get its fps
videos = sorted(glob.glob(os.path.join(DATA_DIR, '*.mp4')))
if not videos:
    raise FileNotFoundError(f"No .mp4 found in {DATA_DIR}")
video_path = videos[0]

cap       = cv2.VideoCapture(video_path)
fps       = cap.get(cv2.CAP_PROP_FPS)
width     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height    = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_fr  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

margin_fr = int((MARGIN_MS / 1000) * fps)

# define our snippet window (clamp to video bounds)
win_start = max(0,             start_frame - margin_fr)
win_end   = min(total_fr - 1,  end_frame   + margin_fr)
snippet_length = win_end - win_start + 1

# 5) Prepare to write out the snippet
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out    = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (width, height))

# 6) Load & sync your gaze stream to frames
rec       = Recording(DATA_DIR)
stream    = rec.sync_gaze_to_video()
gaze_df   = stream.data.dropna(subset=['gaze x [px]', 'gaze y [px]', 'frame_idx'])

# 7) Seek to snippet start
cap.set(cv2.CAP_PROP_POS_FRAMES, win_start)
current = win_start

# 8) Loop and mark
for _ in range(snippet_length):
    ret, frame = cap.read()
    if not ret:
        break

    # find any gaze point for this frame
    pts = gaze_df[gaze_df['frame_idx'] == current]
    for _, row in pts.iterrows():
        x = int(row['gaze x [px]'])
        y = int(row['gaze y [px]'])
        if 0 <= x < width and 0 <= y < height:
            # if within the original saccade window, draw RED
            color = (0, 0, 255) if (start_frame <= current <= end_frame) else None
            if color:
                cv2.circle(frame, (x, y), 8, color, thickness=-1)

    out.write(frame)
    current += 1

# 9) Cleanup
cap.release()
out.release()

print(f"✅ Snippet saved to:\n   {OUTPUT_VIDEO}")


✅ Snippet saved to:
   /Users/biancaschmid/Desktop/saccade_766_snippet.mp4


In [7]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
from pyneon import Recording

# ——— CONFIG ———
DATA_DIR        = '/Users/biancaschmid/Desktop/Spatial-Navigation-Group-1/Real_world_data'
OUTPUT_VIDEO    = '/Users/biancaschmid/Desktop/saccade_766_snippet_slow.mp4'
SACCADE_ID      = 766     # which saccade to highlight
MARGIN_MS       = 100     # milliseconds before/after the saccade
SLOW_MO_FACTOR  = 0.05     # play at 50% speed (0.5 → half speed, 0.25 → quarter speed)
# ——————————

# 1) Load saccades and world‐timestamps
sac_df = pd.read_csv(os.path.join(DATA_DIR, 'saccades.csv'))
wt_df  = pd.read_csv(os.path.join(DATA_DIR, 'world_timestamps.csv'))

# 2) Find saccade 766 timestamps
evt        = sac_df[sac_df['saccade id'] == SACCADE_ID].iloc[0]
start_ts   = evt['start timestamp [ns]']
end_ts     = evt['end timestamp [ns]']

# 3) Map timestamps → frame indices
wt         = wt_df['timestamp [ns]']
start_f    = int((wt - start_ts).abs().idxmin())
end_f      = int((wt - end_ts).abs().idxmin())

# 4) Open video to get fps, dims
videos     = sorted(glob.glob(os.path.join(DATA_DIR, '*.mp4')))
if not videos:
    raise FileNotFoundError(f"No .mp4 found in {DATA_DIR}")
cap        = cv2.VideoCapture(videos[0])
orig_fps   = cap.get(cv2.CAP_PROP_FPS)
width      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_fr   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# compute margins in frames
margin_fr  = int((MARGIN_MS / 1000) * orig_fps)
win_start  = max(0, start_f - margin_fr)
win_end    = min(total_fr - 1, end_f + margin_fr)
snippet_len= win_end - win_start + 1

# 5) Prepare slowed‐down writer
output_fps = orig_fps * SLOW_MO_FACTOR
fourcc     = cv2.VideoWriter_fourcc(*'mp4v')
out        = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, output_fps, (width, height))

# 6) Load & sync gaze stream
rec      = Recording(DATA_DIR)
stream   = rec.sync_gaze_to_video()
gaze_df  = stream.data.dropna(subset=['gaze x [px]', 'gaze y [px]', 'frame_idx'])

# 7) Seek to snippet start
cap.set(cv2.CAP_PROP_POS_FRAMES, win_start)
current = win_start

# 8) Loop and write slowed snippet
for _ in range(snippet_len):
    ret, frame = cap.read()
    if not ret:
        break

    # draw gaze points (red if within saccade window)
    pts = gaze_df[gaze_df['frame_idx'] == current]
    for _, row in pts.iterrows():
        x = int(row['gaze x [px]'])
        y = int(row['gaze y [px]'])
        if 0 <= x < width and 0 <= y < height:
            if start_f <= current <= end_f:
                cv2.circle(frame, (x, y), 8, (0, 0, 255), thickness=-1)

    # write each frame once; playback will be slower
    out.write(frame)
    current += 1

# 9) Cleanup
cap.release()
out.release()

print(f"✅ Slow‐motion snippet saved to:\n   {OUTPUT_VIDEO}\n"
      f"   Original FPS: {orig_fps:.1f}, Output FPS: {output_fps:.1f}")


✅ Slow‐motion snippet saved to:
   /Users/biancaschmid/Desktop/saccade_766_snippet_slow.mp4
   Original FPS: 30.0, Output FPS: 1.5


In [8]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
from pyneon import Recording

# ——— CONFIG ———
DATA_DIR        = '/Users/biancaschmid/Desktop/Spatial-Navigation-Group-1/Real_world_data'
OUTPUT_VIDEO    = '/Users/biancaschmid/Desktop/saccade_746_snippet_slow.mp4'
SACCADE_ID      = 746     # which saccade to highlight
MARGIN_MS       = 100     # milliseconds before/after the saccade
SLOW_MO_FACTOR  = 0.05     # play at 50% speed (0.5 → half speed, 0.25 → quarter speed)
# ——————————

# 1) Load saccades and world‐timestamps
sac_df = pd.read_csv(os.path.join(DATA_DIR, 'saccades.csv'))
wt_df  = pd.read_csv(os.path.join(DATA_DIR, 'world_timestamps.csv'))

# 2) Find saccade 766 timestamps
evt        = sac_df[sac_df['saccade id'] == SACCADE_ID].iloc[0]
start_ts   = evt['start timestamp [ns]']
end_ts     = evt['end timestamp [ns]']

# 3) Map timestamps → frame indices
wt         = wt_df['timestamp [ns]']
start_f    = int((wt - start_ts).abs().idxmin())
end_f      = int((wt - end_ts).abs().idxmin())

# 4) Open video to get fps, dims
videos     = sorted(glob.glob(os.path.join(DATA_DIR, '*.mp4')))
if not videos:
    raise FileNotFoundError(f"No .mp4 found in {DATA_DIR}")
cap        = cv2.VideoCapture(videos[0])
orig_fps   = cap.get(cv2.CAP_PROP_FPS)
width      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_fr   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# compute margins in frames
margin_fr  = int((MARGIN_MS / 1000) * orig_fps)
win_start  = max(0, start_f - margin_fr)
win_end    = min(total_fr - 1, end_f + margin_fr)
snippet_len= win_end - win_start + 1

# 5) Prepare slowed‐down writer
output_fps = orig_fps * SLOW_MO_FACTOR
fourcc     = cv2.VideoWriter_fourcc(*'mp4v')
out        = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, output_fps, (width, height))

# 6) Load & sync gaze stream
rec      = Recording(DATA_DIR)
stream   = rec.sync_gaze_to_video()
gaze_df  = stream.data.dropna(subset=['gaze x [px]', 'gaze y [px]', 'frame_idx'])

# 7) Seek to snippet start
cap.set(cv2.CAP_PROP_POS_FRAMES, win_start)
current = win_start

# 8) Loop and write slowed snippet
for _ in range(snippet_len):
    ret, frame = cap.read()
    if not ret:
        break

    # draw gaze points (red if within saccade window)
    pts = gaze_df[gaze_df['frame_idx'] == current]
    for _, row in pts.iterrows():
        x = int(row['gaze x [px]'])
        y = int(row['gaze y [px]'])
        if 0 <= x < width and 0 <= y < height:
            if start_f <= current <= end_f:
                cv2.circle(frame, (x, y), 8, (0, 0, 255), thickness=-1)

    # write each frame once; playback will be slower
    out.write(frame)
    current += 1

# 9) Cleanup
cap.release()
out.release()

print(f"✅ Slow‐motion snippet saved to:\n   {OUTPUT_VIDEO}\n"
      f"   Original FPS: {orig_fps:.1f}, Output FPS: {output_fps:.1f}")


✅ Slow‐motion snippet saved to:
   /Users/biancaschmid/Desktop/saccade_746_snippet_slow.mp4
   Original FPS: 30.0, Output FPS: 1.5
